# RMIT Hackathon 2026 — Challenge 1 baseline

**Chosen model: `FacebookAI/xlm-roberta-large`** (switch with `MODEL_KEY` below — `mmbert-base` is the other recommended option).

Multilingual 3-class prompt-attack classifier (BENIGN / JAILBREAK / PROMPT_INJECTION), fine-tuned with a **class-weighted cross-entropy** loss, scored with macro-F1.

**Kaggle setup before running**
1. *Add Input* → the competition data `rmit-hackathon-2026`.
2. *Add Input* → *Models* → the approved model you picked (e.g. `aisuko/xlm-roberta-large` or `aisuko/mmbert-base`, framework *Transformers*).
3. *Accelerator* → **GPU T4 x2** or **P100**. Internet can stay off.
4. Rename the notebook to `<university>_<campus>_<team>_C1`, e.g. `RMIT_SGS_myteam_C1`.
5. *Run All* → output `submission.csv` in `/kaggle/working`.

Rough runtime on a T4: xlm-roberta-large ≈ 25–35 min for 3 epochs, mmbert-base ≈ 10–15 min.

In [ ]:
# ==== CONFIG ====
MODEL_KEY = "xlm-roberta-large"  # "xlm-roberta-large" | "xlm-roberta-base" | "mmbert-base" | "mmbert-small"

HF_IDS = {
    "xlm-roberta-large": "FacebookAI/xlm-roberta-large",
    "xlm-roberta-base": "FacebookAI/xlm-roberta-base",
    "mmbert-base": "jhu-clsp/mmBERT-base",
    "mmbert-small": "jhu-clsp/mmBERT-small",
}
LRS = {"xlm-roberta-large": 1e-5, "xlm-roberta-base": 2e-5, "mmbert-base": 3e-5, "mmbert-small": 5e-5}

MAX_LEN = 256          # long prompts are cut head+tail (injections often sit at the end)
HEAD_FRAC = 0.5        # share of the token budget kept from the start of the text
EPOCHS = 3
BATCH_SIZE = 16
LR = LRS[MODEL_KEY]
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
VAL_FRAC = 0.15
CLASS_WEIGHT_POWER = 0.5  # 1.0 = sklearn "balanced" weights, 0.5 = softened, 0.0 = no weighting
TUNE_BIAS = True       # tune per-class logit offsets on the validation split for macro-F1
SEED = 42

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

In [ ]:
import os, random, time, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

transformers.logging.set_verbosity_error()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if USE_AMP else "")

In [ ]:
INPUT = Path("/kaggle/input")

def find_data_file(name):
    hits = sorted(p for p in INPUT.rglob(name) if "rmit" in str(p).lower()) or sorted(INPUT.rglob(name))
    assert hits, f"{name} not found under /kaggle/input — add the competition data as input"
    return hits[0]

def find_model_dir(key):
    hits = sorted(p.parent for p in INPUT.rglob("config.json") if key in str(p).lower())
    if hits:
        return str(hits[0])
    print(f"[warn] no local copy of {key} under /kaggle/input, falling back to the HF hub (needs Internet on)")
    return HF_IDS[key]

TRAIN_CSV = find_data_file("train.csv")
TEST_CSV = find_data_file("test.csv")
MODEL_DIR = find_model_dir(MODEL_KEY)
print("train:", TRAIN_CSV, "\ntest: ", TEST_CSV, "\nmodel:", MODEL_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["text"] = train_df["text"].fillna("").astype(str)
test_df["text"] = test_df["text"].fillna("").astype(str)
train_df["y"] = train_df["label"].map(LABEL2ID)
assert train_df["y"].notna().all(), train_df.loc[train_df["y"].isna(), "label"].unique()

print(train_df.shape, test_df.shape)
display(pd.crosstab(train_df["language"], train_df["label"], margins=True))

In [ ]:
# stratify on label x language so every language keeps its class mix in validation
strat = train_df["label"] + "_" + train_df["language"]
tr_df, va_df = train_test_split(train_df, test_size=VAL_FRAC, stratify=strat, random_state=SEED)
tr_df, va_df = tr_df.reset_index(drop=True), va_df.reset_index(drop=True)
print("train", len(tr_df), "| val", len(va_df))
print(va_df["label"].value_counts())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# work out which special tokens this tokenizer wraps around a sequence (works for XLM-R, mmBERT, DeBERTa, ...)
_full = tokenizer("hello")["input_ids"]
_core = tokenizer("hello", add_special_tokens=False)["input_ids"]
_i = next(i for i in range(len(_full)) if _full[i:i + len(_core)] == _core)
PREFIX, SUFFIX = _full[:_i], _full[_i + len(_core):]
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
print("prefix", PREFIX, "suffix", SUFFIX, "pad", PAD_ID)

def encode(texts):
    budget = MAX_LEN - len(PREFIX) - len(SUFFIX)
    head = int(budget * HEAD_FRAC)
    ids = tokenizer(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    out = []
    for x in ids:
        if len(x) > budget:
            x = x[:head] + x[len(x) - (budget - head):]
        out.append(PREFIX + x + SUFFIX)
    return out

t0 = time.time()
tr_ids, va_ids, te_ids = encode(tr_df["text"]), encode(va_df["text"]), encode(test_df["text"])
lens = np.array([len(x) for x in tr_ids])
print(f"tokenized in {time.time() - t0:.0f}s | median len {np.median(lens):.0f} | "
      f"truncated {np.mean(lens >= MAX_LEN):.1%}")

In [ ]:
class PromptDataset(Dataset):
    def __init__(self, ids, labels=None):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.ids[i], (-1 if self.labels is None else int(self.labels[i]))

def collate(batch):
    n = max(len(x) for x, _ in batch)
    input_ids = torch.full((len(batch), n), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, (x, _) in enumerate(batch):
        input_ids[i, :len(x)] = torch.tensor(x)
        mask[i, :len(x)] = 1
    return input_ids, mask, torch.tensor([y for _, y in batch])

def make_infer_loader(ids):
    # sort by length so padding is minimal, keep the order to undo it afterwards
    order = np.argsort([len(x) for x in ids])
    loader = DataLoader(PromptDataset([ids[i] for i in order]), batch_size=BATCH_SIZE * 2,
                        shuffle=False, collate_fn=collate)
    return loader, order

@torch.no_grad()
def predict_logits(model, ids):
    model.eval()
    loader, order = make_infer_loader(ids)
    chunks = []
    for input_ids, mask, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        chunks.append(logits.float().cpu().numpy())
    out = np.empty((len(ids), len(LABELS)), dtype=np.float32)
    out[order] = np.concatenate(chunks)
    return out

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

In [ ]:
counts = np.bincount(tr_df["y"], minlength=len(LABELS))
class_w = (len(tr_df) / (len(LABELS) * counts)) ** CLASS_WEIGHT_POWER
class_w = class_w / class_w.mean()
print("class weights:", dict(zip(LABELS, class_w.round(3).tolist())))
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device=DEVICE))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # safety-tuned checkpoints ship a 2-class head
).to(DEVICE)

train_loader = DataLoader(PromptDataset(tr_ids, tr_df["y"].values), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate, drop_last=True)
no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight"]
params = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(params, lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"{total_steps} steps, lr {LR}")

In [ ]:
best_f1, best_state, best_val_logits = -1.0, None, None
y_va = va_df["y"].values

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, running = time.time(), 0.0
    for step, (input_ids, mask, y) in enumerate(train_loader, 1):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        loss = loss_fn(logits.float(), y.to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {running / step:.4f} ({time.time() - t0:.0f}s)")

    val_logits = predict_logits(model, va_ids)
    f1 = macro_f1(y_va, val_logits.argmax(1))
    print(f"epoch {epoch}: train loss {running / len(train_loader):.4f} | val macro-F1 {f1:.4f} | {time.time() - t0:.0f}s")
    if f1 > best_f1:
        best_f1, best_val_logits = f1, val_logits
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"best val macro-F1 {best_f1:.4f}")
if best_f1 < 0.6:
    print("[warn] training looks collapsed (large models sometimes do) — rerun with another SEED or a lower LR")

In [ ]:
va_pred = best_val_logits.argmax(1)
print(classification_report(y_va, va_pred, target_names=LABELS, digits=4, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_va, va_pred), index=LABELS, columns=LABELS))
print("\nper-language macro-F1:")
for lang, g in va_df.groupby("language"):
    print(f"  {lang}: {macro_f1(g['y'], va_pred[g.index]):.4f}  (n={len(g)})")

In [ ]:
# Per-class logit offsets tuned on validation. Only kept if they clearly help,
# since ~150 PROMPT_INJECTION rows in val is not much to tune on.
bias = np.zeros(len(LABELS), dtype=np.float32)
if TUNE_BIAS:
    grid = np.arange(-2.0, 2.01, 0.25)
    best_b, best_b_f1 = bias.copy(), best_f1
    for bj in grid:
        for bp in grid:
            b = np.array([0.0, bj, bp], dtype=np.float32)
            f = macro_f1(y_va, (best_val_logits + b).argmax(1))
            if f > best_b_f1 + 1e-9:
                best_b, best_b_f1 = b, f
    if best_b_f1 - best_f1 >= 0.002:
        bias = best_b
    print(f"bias {dict(zip(LABELS, bias.tolist()))} | val macro-F1 {best_f1:.4f} -> {macro_f1(y_va, (best_val_logits + bias).argmax(1)):.4f}")

In [ ]:
test_logits = predict_logits(model, te_ids)
test_pred = (test_logits + bias).argmax(1)

sub = pd.DataFrame({"id": test_df["id"], "label": [LABELS[i] for i in test_pred]})
assert len(sub) == len(test_df) == 6000, len(sub)
assert sub["id"].is_unique and set(sub["label"]) <= set(LABELS)
sub.to_csv("/kaggle/working/submission.csv", index=False)

# keep raw logits around for ensembling later
np.save(f"/kaggle/working/val_logits_{MODEL_KEY}.npy", best_val_logits)
np.save(f"/kaggle/working/test_logits_{MODEL_KEY}.npy", test_logits)
va_df[["id"]].to_csv("/kaggle/working/val_ids.csv", index=False)

print(sub["label"].value_counts(normalize=True).round(3))
sub.head()